# Exploracao dos Dados BCB

Notebook para verificar integridade dos dados coletados do Banco Central do Brasil e outras fontes.

**Dados disponiveis:**
- **SGS Daily**: Selic, CDI, Dolar PTAX, Euro PTAX
- **SGS Monthly**: IBC-Br Bruto, IBC-Br Dessazonalizado, IGP-M
- **Expectations**: Expectativas do Relatorio Focus (IPCA, IGP-M, PIB, Cambio, Selic)
- **IPEA**: Dados agregados de emprego (CAGED saldo, admissoes, desligamentos, taxa desemprego)
- **MTE CAGED**: Microdados de movimentacao de emprego formal

## 1. Setup e Carregamento

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# MTE CAGED
from mte.caged import CAGEDCollector

# Configurar matplotlib
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 10

# Caminho para dados
data_path = Path.cwd().parent / 'data'
processed_path = data_path / 'processed'

# Inicializar collectors
caged_collector = CAGEDCollector(data_path)

print(f"Data path: {data_path}")

In [ ]:
# Carregar todos os dados consolidados
df_sgs_daily = pd.read_parquet(processed_path / 'bacen_sgs_daily_consolidated.parquet')
df_sgs_monthly = pd.read_parquet(processed_path / 'bacen_sgs_monthly_consolidated.parquet')
df_expectations = pd.read_parquet(processed_path / 'expectations_consolidated.parquet')

print("Dados carregados com sucesso!")
print(f"  SGS Daily: {df_sgs_daily.shape[0]:,} registros x {df_sgs_daily.shape[1]} colunas")
print(f"  SGS Monthly: {df_sgs_monthly.shape[0]:,} registros x {df_sgs_monthly.shape[1]} colunas")
print(f"  Expectations: {df_expectations.shape[0]:,} registros x {df_expectations.shape[1]} colunas")

## 2. SGS Daily

In [ ]:
print("=" * 60)
print("SGS DAILY")
print("=" * 60)
print(f"\nShape: {df_sgs_daily.shape}")
print(f"Colunas: {list(df_sgs_daily.columns)}")
print(f"Periodo: {df_sgs_daily.index.min()} a {df_sgs_daily.index.max()}")
print(f"\nValores faltantes:")
print(df_sgs_daily.isna().sum())

In [ ]:
print("Primeiros registros:")
display(df_sgs_daily.head())
print("\nUltimos registros:")
display(df_sgs_daily.tail())

## 3. SGS Monthly

In [ ]:
print("=" * 60)
print("SGS MONTHLY")
print("=" * 60)
print(f"\nShape: {df_sgs_monthly.shape}")
print(f"Colunas: {list(df_sgs_monthly.columns)}")
print(f"Periodo: {df_sgs_monthly.index.min()} a {df_sgs_monthly.index.max()}")
print(f"\nValores faltantes:")
print(df_sgs_monthly.isna().sum())

In [ ]:
print("Primeiros registros:")
display(df_sgs_monthly.head())
print("\nUltimos registros:")
display(df_sgs_monthly.tail())

## 4. Expectations

In [ ]:
print("=" * 60)
print("EXPECTATIONS")
print("=" * 60)
print(f"\nShape: {df_expectations.shape}")
print(f"Colunas: {list(df_expectations.columns)}")
print(f"Periodo: {df_expectations.index.min()} a {df_expectations.index.max()}")
print(f"\nValores faltantes:")
print(df_expectations.isna().sum())

In [ ]:
# Contagem por fonte (_source)
print("Registros por indicador (_source):")
print(df_expectations['_source'].value_counts().sort_index())

In [ ]:
print("Primeiros registros:")
display(df_expectations.head())
print("\nUltimos registros:")
display(df_expectations.tail())

## 5. Cobertura Temporal

In [ ]:
def cobertura_temporal(df, nome):
    """Analisa a cobertura temporal de cada coluna."""
    print(f"\n{nome}")
    print("-" * 70)
    for col in df.columns:
        if col == '_source':
            continue
        dados = df[col].dropna()
        if len(dados) > 0:
            try:
                print(f"{col:30} | {dados.index.min().strftime('%Y-%m-%d')} a {dados.index.max().strftime('%Y-%m-%d')} | {len(dados):>8,} registros")
            except Exception:
                print(f"{col:30} | {len(dados):>8,} registros")
        else:
            print(f"{col:30} | SEM DADOS")

cobertura_temporal(df_sgs_daily, "SGS DAILY")
cobertura_temporal(df_sgs_monthly, "SGS MONTHLY")

In [ ]:
# Cobertura temporal por fonte de expectations
print("\nEXPECTATIONS (por _source)")
print("-" * 70)

for source in sorted(df_expectations['_source'].unique()):
    subset = df_expectations[df_expectations['_source'] == source]
    min_date = subset.index.min().strftime('%Y-%m-%d')
    max_date = subset.index.max().strftime('%Y-%m-%d')
    print(f"{source:30} | {min_date} a {max_date} | {len(subset):>8,} registros")

## 6. Visualizacoes - SGS

In [ ]:
# Selic e CDI
fig, ax = plt.subplots(figsize=(14, 5))

start_date = df_sgs_daily['selic'].first_valid_index()
df_plot = df_sgs_daily.loc[start_date:]

if 'selic' in df_plot.columns:
    df_plot['selic'].dropna().plot(ax=ax, label='Meta Selic', linewidth=1)
if 'cdi_anualizado' in df_plot.columns:
    df_plot['cdi_anualizado'].dropna().plot(ax=ax, label='CDI Anualizado', linewidth=0.5, alpha=0.7)

ax.set_title('Taxa Selic e CDI')
ax.set_xlabel('Data')
ax.set_ylabel('Taxa (% a.a.)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Dolar e Euro PTAX
fig, ax = plt.subplots(figsize=(14, 5))

start_date = df_sgs_daily['euro_ptax'].first_valid_index()
df_plot = df_sgs_daily.loc[start_date:]

if 'dolar_ptax' in df_plot.columns:
    df_plot['dolar_ptax'].dropna().plot(ax=ax, label='Dolar PTAX', linewidth=1)
if 'euro_ptax' in df_plot.columns:
    df_plot['euro_ptax'].dropna().plot(ax=ax, label='Euro PTAX', linewidth=1)

ax.set_title('Taxas de Cambio PTAX')
ax.set_xlabel('Data')
ax.set_ylabel('R$')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# IBC-Br
fig, ax = plt.subplots(figsize=(14, 5))

if 'ibc_br_bruto' in df_sgs_monthly.columns:
    df_sgs_monthly['ibc_br_bruto'].dropna().plot(ax=ax, label='IBC-Br Bruto', linewidth=1)
if 'ibc_br_dessaz' in df_sgs_monthly.columns:
    df_sgs_monthly['ibc_br_dessaz'].dropna().plot(ax=ax, label='IBC-Br Dessazonalizado', linewidth=1)

ax.set_title('IBC-Br - Indice de Atividade Economica')
ax.set_xlabel('Data')
ax.set_ylabel('Indice')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# IGP-M
fig, ax = plt.subplots(figsize=(14, 5))

df_plot = df_sgs_monthly.loc['2000':]

if 'igp_m' in df_plot.columns:
    df_plot['igp_m'].dropna().plot(ax=ax, color='green', linewidth=1)

ax.axhline(y=0, color='red', linestyle='--', linewidth=0.8, alpha=0.7)
ax.set_title('IGP-M - Variacao Mensal (%)')
ax.set_xlabel('Data')
ax.set_ylabel('Variacao (%)')
plt.tight_layout()
plt.show()

## 7. Visualizacoes - Expectations

In [ ]:
# Distribuicao de registros por fonte
fig, ax = plt.subplots(figsize=(12, 5))

counts = df_expectations['_source'].value_counts().sort_index()
counts.plot(kind='barh', ax=ax, color='steelblue')

ax.set_title('Registros por Indicador (Expectations)')
ax.set_xlabel('Quantidade de Registros')
ax.set_ylabel('Indicador')

for i, v in enumerate(counts.values):
    ax.text(v + 1000, i, f'{v:,}', va='center')

plt.tight_layout()
plt.show()

In [ ]:
# Exemplo: IPCA Anual - Mediana ao longo do tempo
if 'ipca_anual' in df_expectations['_source'].values:
    ipca_anual = df_expectations[df_expectations['_source'] == 'ipca_anual'].copy()
    
    if 'Mediana' in ipca_anual.columns:
        mediana_diaria = ipca_anual.groupby(ipca_anual.index)['Mediana'].mean()
        
        fig, ax = plt.subplots(figsize=(14, 5))
        mediana_diaria.plot(ax=ax, linewidth=0.5)
        
        ax.set_title('IPCA Anual - Mediana das Expectativas')
        ax.set_xlabel('Data da Pesquisa')
        ax.set_ylabel('Expectativa (%)')
        plt.tight_layout()
        plt.show()
        
        print(f"Periodo: {ipca_anual.index.min()} a {ipca_anual.index.max()}")
        print(f"Registros: {len(ipca_anual):,}")

In [ ]:
# Exemplo: Selic - Mediana ao longo do tempo
if 'selic' in df_expectations['_source'].values:
    selic_exp = df_expectations[df_expectations['_source'] == 'selic'].copy()
    
    if 'Mediana' in selic_exp.columns:
        mediana_diaria = selic_exp.groupby(selic_exp.index)['Mediana'].mean()
        
        fig, ax = plt.subplots(figsize=(14, 5))
        mediana_diaria.plot(ax=ax, linewidth=0.5)
        
        ax.set_title('Selic - Mediana das Expectativas')
        ax.set_xlabel('Data da Pesquisa')
        ax.set_ylabel('Expectativa (%)')
        plt.tight_layout()
        plt.show()
        
        print(f"Periodo: {selic_exp.index.min()} a {selic_exp.index.max()}")
        print(f"Registros: {len(selic_exp):,}")

## 8. Resumo

Este notebook verificou a integridade dos dados coletados:

- **SGS Daily**: Series diarias de taxas de juros e cambio
- **SGS Monthly**: Series mensais de atividade economica e inflacao
- **Expectations**: Expectativas de mercado do Relatorio Focus

Para analises mais detalhadas (correlacoes, tendencias, previsoes), crie notebooks especificos.

## 9. IPEA - Dados Agregados de Emprego

In [ ]:
# Carregar dados IPEA consolidados
df_ipea = pd.read_parquet(processed_path / 'ipea_monthly_consolidated.parquet')

print("=" * 60)
print("IPEA - DADOS AGREGADOS DE EMPREGO")
print("=" * 60)
print(f"\nShape: {df_ipea.shape}")
print(f"Colunas: {list(df_ipea.columns)}")
print(f"Periodo: {df_ipea.index.min()} a {df_ipea.index.max()}")
print(f"\nValores faltantes:")
print(df_ipea.isna().sum())

In [ ]:
print("Primeiros registros:")
display(df_ipea.head())
print("\nUltimos registros:")
display(df_ipea.tail())

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

df_ipea['caged_saldo'].dropna().plot(ax=ax, linewidth=1, color='steelblue')
ax.axhline(y=0, color='red', linestyle='--', linewidth=0.8, alpha=0.7)
ax.set_title('Saldo de Empregos Formais (CAGED)')
ax.set_xlabel('Data')
ax.set_ylabel('Saldo (pessoas)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x/1000:.0f}k'))

plt.tight_layout()
plt.show()

In [ ]:
df_plot = df_ipea[['caged_admissoes', 'caged_desligamentos']].dropna()

fig, ax = plt.subplots(figsize=(14, 5))

df_plot['caged_admissoes'].plot(ax=ax, label='Admissoes', linewidth=1)
df_plot['caged_desligamentos'].plot(ax=ax, label='Desligamentos', linewidth=1)

ax.set_title('Admissoes vs Desligamentos (CAGED)')
ax.set_xlabel('Data')
ax.set_ylabel('Pessoas')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x/1e6:.1f}M'))
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

df_ipea['taxa_desemprego'].dropna().plot(ax=ax, linewidth=1, color='darkred', marker='o', markersize=2)

ax.set_title('Taxa de Desemprego (PNAD Continua)')
ax.set_xlabel('Data')
ax.set_ylabel('Taxa (%)')

plt.tight_layout()
plt.show()

## 10. MTE CAGED - Microdados

In [ ]:
# Carregar dados de um mes usando CAGEDCollector
df_caged = caged_collector.read(year=2025, months=[10])

print("=" * 60)
print("MTE CAGED - MICRODADOS (2025-10)")
print("=" * 60)
print(f"\nShape: {df_caged.shape}")
print(f"\nColunas disponiveis:")
for i, col in enumerate(df_caged.columns, 1):
    print(f"  {i:2d}. {col}")

In [ ]:
print("Tipos de dados:")
print(df_caged.dtypes)
print("\nAmostra de 5 registros:")
display(df_caged.sample(5))

In [ ]:
# Top 10 UFs por movimentacao
uf_counts = df_caged.groupby('uf')['saldomovimentação'].agg(['sum', 'count'])
uf_counts.columns = ['saldo', 'movimentacoes']
uf_counts = uf_counts.sort_values('movimentacoes', ascending=False).head(10)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(uf_counts.index.astype(str), uf_counts['movimentacoes'], color='steelblue')
axes[0].set_title('Top 10 UFs - Movimentacoes (2025-10)')
axes[0].set_xlabel('Quantidade')
axes[0].invert_yaxis()

colors = ['green' if x > 0 else 'red' for x in uf_counts['saldo']]
axes[1].barh(uf_counts.index.astype(str), uf_counts['saldo'], color=colors)
axes[1].set_title('Top 10 UFs - Saldo (2025-10)')
axes[1].set_xlabel('Saldo')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# Saldo por secao economica
secao_saldo = df_caged.groupby('seção')['saldomovimentação'].sum().sort_values()

fig, ax = plt.subplots(figsize=(14, 6))

colors = ['green' if x > 0 else 'red' for x in secao_saldo]
secao_saldo.plot(kind='barh', ax=ax, color=colors)
ax.set_title('Saldo por Secao Economica (CNAE) - 2025-10')
ax.set_xlabel('Saldo')
ax.axvline(x=0, color='black', linestyle='-', linewidth=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Criar faixas etarias
bins = [0, 18, 25, 35, 45, 55, 65, 100]
labels = ['<18', '18-24', '25-34', '35-44', '45-54', '55-64', '65+']
df_caged['faixa_etaria'] = pd.cut(df_caged['idade'], bins=bins, labels=labels)

faixa_stats = df_caged.groupby('faixa_etaria', observed=False)['saldomovimentação'].agg(['sum', 'count'])
faixa_stats.columns = ['saldo', 'movimentacoes']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(faixa_stats.index.astype(str), faixa_stats['movimentacoes'], color='steelblue')
axes[0].set_title('Movimentacoes por Faixa Etaria (2025-10)')
axes[0].set_xlabel('Faixa Etaria')
axes[0].set_ylabel('Quantidade')

colors = ['green' if x > 0 else 'red' for x in faixa_stats['saldo']]
axes[1].bar(faixa_stats.index.astype(str), faixa_stats['saldo'], color=colors)
axes[1].set_title('Saldo por Faixa Etaria (2025-10)')
axes[1].set_xlabel('Faixa Etaria')
axes[1].set_ylabel('Saldo')

plt.tight_layout()
plt.show()